# UCB-FE Experiments

This notebooks provides  neccessary calculations for the UCB-FE experiments.
Logic about using ML models are implemeted in the [utils file](./utils_ucb_fe.py).

Yoy should run this notebook after notebook 2.

In [ ]:
import numpy  as np
import pandas as pd
from sklearn.metrics import roc_auc_score

In [ ]:
#params
cold_periods = [0, 10, 100, 200, 500]
delta = 1.5
top_n = 10
tail_m = 30

In [ ]:
# Read data
df =  pd.read_parquet('data/results/df_ucb_fe.parquet')

model_names = ['catboost', 'lightgbm', 'xgboost' 'tabnet']

save_columns = ['target', 'request_id', 'item_id']

for name in model_names:
    save_columns.append('base_ctr_pred_' + name)
    for T in cold_periods:
        save_columns.append('ucb_ctr_pred_T_'+str(T) + '_' + name)

df = df[save_columns]

In [ ]:

import warnings
warnings.filterwarnings("ignore")

class StratifiedAUC:
        """
        Weighted average of ROC-AUC metric by group, where weights are determined by the number of positive targets in each group. 
        StratifiedAUC is taken from a Yahoo article https://arxiv.org/abs/2312.05052
        """
    
        default_name = "StratifiedAUC"
    
        def __init__(
            self,
            target_column: str,
            group_column: str,
        ):
            self.target_column = target_column
            self.group_column = group_column
    
        def __call__(self, serp: pd.DataFrame, rank_column: str) -> float:
            if serp[self.target_column].sum() < 1:
                return np.nan
    
            def _fn_num(group_df: pd.DataFrame) -> float:
                if group_df[self.target_column].nunique() == 1:
                    return np.nan
                roc_auc = roc_auc_score(group_df[self.target_column], group_df[rank_column])
                return roc_auc * group_df[self.target_column].sum()
    
            def _fn_den(group_df: pd.DataFrame) -> float:
                if group_df[self.target_column].nunique() == 1:
                    return 0
                return group_df[self.target_column].sum()
    
            num = serp.groupby(self.group_column).apply(_fn_num).sum()
            den = serp.groupby(self.group_column).apply(_fn_den).sum()
    
            return num / den
    
        @property
        def name(self):
            return self.default_name
    
strat_auc_score = StratifiedAUC('target', 'request_id')
df_grouped = df.groupby('request_id')

auc_data = []
strat_auc_data = []

columns_auc = ['AUC base']
columns_strat_auc = ['Stratified AUC base']

for T in cold_periods:
    columns_auc.append('AUC fe-ucb, T = ' + str(T))
    columns_strat_auc.append('Stratified AUC fe-ucb, T = ' + str(T))

for name in model_names:
    auc = []
    strat_auc= []

    auc_old = roc_auc_score(df['target'], df['base_ctr_pred_' + name])
    strat_auc_old = df_grouped.apply(strat_auc_score, rank_column= 'base_ctr_pred_' + name)

    auc.append(auc_old)
    strat_auc.append(strat_auc_old.mean())

    for T in cold_periods:
        print("period = ", T)
        auc_new= roc_auc_score(df['target'], df['ucb_ctr_pred_T_'+str(T) + '_' + name])    
        strat_auc_new = df_grouped.apply(strat_auc_score, rank_column= 'ucb_ctr_pred_T_'+str(T) + '_' + name)
    
        auc.append(auc_new)
        strat_auc.append(strat_auc_new.mean())
    
    auc_data.append(auc)
    strat_auc_data.append(strat_auc)

result_auc = pd.DataFrame(
    data = auc_data, 
    columns= columns_auc, 
    index= model_names
)

result_strat_auc = pd.DataFrame(
    data = strat_auc_data, 
    columns= columns_strat_auc, 
    index= model_names
)

In [ ]:
result_auc.to_csv('data/results/auc.csv', index = True)
result_auc

In [ ]:
result_strat_auc.to_csv('data/results/strat_auc.csv', index = True)
result_strat_auc